# Reallocation analysis

Measures how synthetic travel activity is reallocated across transport modes and population segments. The notebook expects generated panel files and prepared feature lists and produces grouped summaries and visualisations.

In [ ]:
# Load necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from collections import defaultdict
import warnings
warnings.filterwarnings("ignore")

# Set display options for pandas
pd.set_option('display.max_columns', None)

In [ ]:
# Load generated data
path = './'
pop_size = 4
model = 'best_cvae_model'
var = 100

df_samples_con = pd.read_parquet(path + f"data\TUmodeldata_panel_ally_{pop_size}.parquet")  # Load as DataFrame

# Load feature lists
with open(path +"data/pref_cat.txt", "r") as f:
    cat_features = f.read().splitlines()
with open(path +"data/pref_num.txt", "r") as f:
    num_features = f.read().splitlines()
with open(path +"data/con_num.txt", "r") as f:
    num_cond_features = f.read().splitlines()
with open(path +"data/con_cat.txt", "r") as f:
    cat_cond_features = f.read().splitlines()

# List of trips columns and average length columns
trips_columns = cat_features[1:]
avg_length_columns = num_features

In [ ]:
def read_parquet_file_pandas(file, df_samples_con, group=None):
    df = pd.read_parquet(file)  # Read as a pandas DataFrame

    # Calculate mileage per mode
    trcol = ['trips_mode_1', 'trips_mode_2', 'trips_mode_3', 'trips_mode_4', 'trips_mode_5', 'trips_mode_6', 'trips_mode_7']
    lencol = ['lengths_mode_1', 'lengths_mode_2', 'lengths_mode_3', 'lengths_mode_4', 'lengths_mode_5', 'lengths_mode_6', 'lengths_mode_7']
    for i in range(len(trips_columns)):
        mileage_column = f'mileage_mode_{i+1}'  # Create a new column name for mileage
        df[mileage_column] = df[trcol[i]] * df[lencol[i]]

    # Reset the index to ensure known divisions
    df = df.reset_index(drop=True)

    # Reset the index for both DataFrames to ensure known divisions
    df = df.reset_index()
    df_samples_con = df_samples_con.reset_index()


    def assign_quartiles(group):
        # Compute year-specific quartiles
        q = group['access_pop_exp_car'].quantile([0.25, 0.5, 0.75])
        
        # Define bins using these quartiles
        bins = [
            group['access_pop_exp_car'].min() - 1,
            q.loc[0.25],
            q.loc[0.5],
            q.loc[0.75],
            group['access_pop_exp_car'].max() + 1
        ]
        
        # Assign quartile labels
        group['access_pop_exp_car'] = pd.cut(
            group['access_pop_exp_car'], 
            bins=bins, 
            labels=['Q1', 'Q2', 'Q3', 'Q4'],
            include_lowest=True
        )
        
        return group

    if group == 'access_pop_exp_car':
        # Apply per-year
        df_samples_con = df_samples_con.groupby('DiaryYear', group_keys=False).apply(assign_quartiles)
    
    if group == 'RespAgeCorrect':
        # Age quantiles
        quantiles = df_samples_con['RespAgeCorrect'].quantile([0.25, 0.5, 0.75])
        bins= [df_samples_con['RespAgeCorrect'].min()-1, quantiles[0.25], quantiles[0.5], quantiles[0.75], df_samples_con['RespAgeCorrect'].max()+1]
        labels = ['Q1', 'Q2', 'Q3', 'Q4']
        df_samples_con['RespAgeCorrect'] = pd.cut(df_samples_con['RespAgeCorrect'], bins=bins, labels=labels) 

    if group == 'IncFamily2023':
        # Create income quartiles
        quantiles = df_samples_con['IncFamily2023'].quantile([0.25, 0.5, 0.75])
        bins= [df_samples_con['IncFamily2023'].min()-1, quantiles[0.25], quantiles[0.5], quantiles[0.75], df_samples_con['IncFamily2023'].max()+1]
        labels = ['low', 'medium', 'high', 'very_high']
        df_samples_con['IncFamily2023'] = pd.cut(df_samples_con['IncFamily2023'], bins=bins, labels=labels) 

    if group == 'HousehNumcars':
        # Create a new column for the number of cars in the household, capping it at 3
        df_samples_con['HousehNumcars'] = df_samples_con['HousehNumcars'].apply(lambda x: 3 if x > 3 else x)

    # Concatenate the two DataFrames
    df_combined = pd.concat([df, df_samples_con], axis=1)
    
    if group == 'purposes':

        # Map the 'trip_purpose' column to broader categories
        map = {
        0: 'Leisure/Other',
        4: 'Work/Education/Home',
        6: 'Work/Education/Home',
        2: 'Leisure/Other',
        3: 'Leisure/Other',
        1: 'Leisure/Other',
        7: 'Leisure/Other',
        5: 'Leisure/Other'
        }

        df_combined['purposes'] = df_combined['purposes'].map(map)

    return df_combined

In [ ]:
MILEAGE_COLS = [
    'mileage_mode_1',
    'mileage_mode_2',
    'mileage_mode_3',
    'mileage_mode_4',
    'mileage_mode_5',
    'mileage_mode_6',
    'mileage_mode_7'
]

MODE_LABELS = {
    'mileage_mode_1': 'Walk',
    'mileage_mode_2': 'Bike',
    'mileage_mode_3': 'E-bike',
    'mileage_mode_4': 'Car',
    'mileage_mode_5': 'Car pas.',
    'mileage_mode_6': 'Pub.tr.',
    'mileage_mode_7': 'Moped'
}

In [ ]:
# Path setup ---
directory_path = os.path.join(path, f"results/samples_{pop_size}_{model}/")

# List all .parquet files ---
parquet_files = [
    os.path.join(directory_path, f)
    for f in os.listdir(directory_path)
    if f.endswith(".parquet")
]  # Limit to first 10 files for testing

In [ ]:
def compute_rank_modes(
    df,
    mileage_cols,
    add_no_mode=True
):

    arr = df[mileage_cols].to_numpy()

    order = np.argsort(-arr, axis=1)

    active_modes = (arr > 0).sum(axis=1)

    rank_cols = {}

    n_modes = len(mileage_cols)

    for rank in range(n_modes):

        col = f"rank_{rank+1}"

        values = np.array(mileage_cols)[
            order[:, rank]
        ]

        values = pd.Series(
            values,
            index=df.index,
            dtype="object"
        )

        insufficient_modes = (
            active_modes < (rank + 1)
        )

        if add_no_mode:
            values[insufficient_modes] = "No mode"
        else:
            values[insufficient_modes] = np.nan

        rank_cols[col] = values

    return pd.DataFrame(rank_cols)


In [ ]:
def build_transition_matrix(
    df,
    rank_col,
    year_start=2015,
    year_end=2024
):

    pivot = df.pivot_table(
        index="NewSessionId",
        columns="DiaryYear",
        values=rank_col,
        aggfunc="first"
    )

    pivot = pivot.dropna(
        subset=[year_start, year_end]
    )

    tm = (
        pivot
        .groupby([year_start, year_end])
        .size()
        .unstack(fill_value=0)
    )

    return tm

In [ ]:
def build_rank_transition_matrices(
    df,
    max_rank
):

    matrices = {}

    for rank in range(1, max_rank + 1):

        matrices[rank] = build_transition_matrix(
            df,
            f"rank_{rank}"
        )

    return matrices

In [ ]:
def compute_Nk(
    rank_matrices,
    focus_mode="mileage_mode_3"
):

    Nk = 0

    for rank, tm in rank_matrices.items():

        if focus_mode not in tm.index:
            continue

        Nk += (
            tm[focus_mode].sum()
            - tm.loc[focus_mode].sum()
            + tm.loc[focus_mode, focus_mode]
        )

    return Nk

In [ ]:
def compute_reallocation(
    rank_matrices,
    focus_mode="mileage_mode_3"
):
    
    all_modes = sorted(
    set().union(
        *[tm.index for tm in rank_matrices.values()],
        *[tm.columns for tm in rank_matrices.values()]
    )
    )

    for rank in rank_matrices:

        rank_matrices[rank] = (
            rank_matrices[rank]
            .reindex(
                index=all_modes,
                columns=all_modes,
                fill_value=0
            )
        )

    Nk = compute_Nk(
        rank_matrices,
        focus_mode
    )

    results = {}

    for rank, tm in rank_matrices.items():

        rank_result = {}

        for mode in tm.index:

            flow = (
                tm.loc[mode, focus_mode]
                - tm.loc[focus_mode, mode]
            )

            rank_result[mode] = (
                flow / Nk * 100
            )

        rank_result[focus_mode] = (
            tm.loc[focus_mode, focus_mode]
            / Nk * 100
        )

        results[rank] = pd.Series(
            rank_result
        )

    return results

In [ ]:
def process_simulation(
    file_path,
    group_col=None,
    max_rank=7
):

    ddf = read_parquet_file_pandas(
        file_path,
        df_samples_con,
        group=group_col
    )

    ddf = ddf[
        ddf["DiaryYear"].isin(
            [2015, 2024]
        )
    ].copy()

    rank_df = compute_rank_modes(
        ddf,
        mileage_cols=MILEAGE_COLS
    )

    ddf = pd.concat(
        [ddf, rank_df],
        axis=1
    )

    if group_col is None:

        groups = {
            "All": ddf
        }

    else:

        groups = {
            g: x.copy()
            for g, x
            in ddf.groupby(group_col)
        }

    output = {}

    for group, gdf in groups.items():

        matrices = build_rank_transition_matrices(
            gdf,
            max_rank=max_rank
        )

        realloc = compute_reallocation(
            matrices,
            focus_mode="mileage_mode_3"
        )

        output[group] = realloc

    return output

In [ ]:
def summarize_reallocation(
    group_summary,
    include_ebike=False,
    split_no_mode=True
):

    total = pd.Series(dtype=float)

    # Sum across ranks
    for rank in group_summary.keys():

        total = total.add(
            group_summary[rank]["mean"],
            fill_value=0
        )

    # --------------------------------------------------
    # Split No mode
    # --------------------------------------------------

    if split_no_mode:

        no_mode_primary = (
            group_summary
            .get(1, {})
            .get("mean", pd.Series())
            .get("No mode", 0)
        )

        no_mode_secondary_plus = sum(
            group_summary[r]["mean"].get("No mode", 0)
            for r in group_summary
            if r >= 2
        )

        total = total.drop(
            "No mode",
            errors="ignore"
        )

        total["No mode (Primary)"] = no_mode_primary
        total["No mode (Secondary+)"] = (
            no_mode_secondary_plus
        )

    rename_dict = {
        "mileage_mode_1": "Walk",
        "mileage_mode_2": "Bike",
        "mileage_mode_3": "E-bike",
        "mileage_mode_4": "Car",
        "mileage_mode_5": "Car pas.",
        "mileage_mode_6": "Pub.tr.",
        "mileage_mode_7": "Moped"
    }

    total = total.rename(index=rename_dict)

    if not include_ebike:
        total = total.drop(
            "E-bike",
            errors="ignore"
        )

    # fixed paper order
    desired_order = [
        "No mode (Primary)",
        "No mode (Secondary+)",
        "Walk",
        "E-bike",
        "Bike",
        "Moped",
        "Car",
        "Car pas.",
        "Pub.tr."        
    ]

    desired_order = [
        x for x in desired_order
        if x in total.index
    ]

    table = pd.DataFrame({
        "Contribution (%)":
            total.reindex(desired_order)
    })

    table["Contribution (%)"] = (
        table["Contribution (%)"]
        .round(1)
    )

    return table

In [ ]:
all_results = defaultdict(
    lambda:
    defaultdict(list)
)

for file in parquet_files:

    res = process_simulation(
        file,
        group_col=None,
        max_rank=7
    )

    for group in res:

        for rank in res[group]:

            all_results[group][rank].append(
                res[group][rank]
            )

In [ ]:
summary = {}

for group in all_results:

    summary[group] = {}

    for rank in all_results[group]:

        df_rank = pd.concat(
            all_results[group][rank],
            axis=1
        )

        summary[group][rank] = {
            "mean": df_rank.mean(axis=1),
            "std": df_rank.std(axis=1)
        }

In [ ]:
mode_labels = {
    'mileage_mode_1': 'Walk',
    'mileage_mode_2': 'Bike',
    'mileage_mode_3': 'E-bike',
    'mileage_mode_4': 'Car',
    'mileage_mode_5': 'Car pas.',
    'mileage_mode_6': 'Pub.tr.',
    'mileage_mode_7': 'Moped',
    'No mode': 'No mode'
}

desired_order = [
    'Walk',
    'Bike',
    'E-bike',
    'Moped',
    'Car',
    'Car pas.',
    'Pub.tr.',
    'No mode'
]

In [ ]:
rank1 = (
    summary[group][1]['mean']
    .rename(index=mode_labels)
    .reindex(desired_order, fill_value=0)
)

rank2 = (
    summary[group][2]['mean']
    .rename(index=mode_labels)
    .reindex(desired_order, fill_value=0)
)

rank3 = (
    summary[group][3]['mean']
    .rename(index=mode_labels)
    .reindex(desired_order, fill_value=0)
)

rank4plus = pd.Series(
    0,
    index=desired_order,
    dtype=float
)

for r in summary[group]:

    if r >= 4:

        rank4plus += (
            summary[group][r]['mean']
            .rename(index=mode_labels)
            .reindex(desired_order, fill_value=0)
        )

In [ ]:
table = summarize_reallocation(
    summary['All'],   # <- group 1
    include_ebike=True
)

print(table)

print(
    "\nTotal:",
    table["Contribution (%)"].sum()
)

In [ ]:
x = np.arange(len(desired_order))

fig, ax = plt.subplots(
    figsize=(12, 6)
)

# Primary
ax.bar(
    x,
    rank1,
    label='Primary',
    color='#6baed6',
    alpha=0.8,
    edgecolor='black'
)

# Secondary
ax.bar(
    x,
    rank2,
    bottom=rank1,
    label='Secondary',
    color='#fdae6b',
    alpha=0.8,
    edgecolor='black'
)

# Tertiary
ax.bar(
    x,
    rank3,
    bottom=rank1 + rank2,
    label='Tertiary',
    color='#74c476',
    alpha=0.8,
    edgecolor='black'
)

# 4th+
ax.bar(
    x,
    rank4plus,
    bottom=rank1 + rank2 + rank3,
    label='4th+',
    color='#9e9ac8',
    alpha=0.8,
    edgecolor='black'
)

# Formatting
ax.set_xticks(x)
ax.set_xticklabels(
    desired_order,
    rotation=45,
    ha='right',
    fontsize=12
)

ax.axhline(
    0,
    color='black',
    linewidth=0.8,
    linestyle='--',
    alpha=0.3
)

ax.set_ylabel(
    'Contribution to E-bike growth (%)'
)

ax.set_title(
    'E-bike reallocation'
)

ax.legend(
    frameon=True,
    ncol=4,
    fontsize=10
)

ax.grid(
    axis='y',
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    path + f"results/plots/samples_{pop_size}_{model}_reallocation_plot.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

## By Segment

In [ ]:
def run_reallocation_analysis(
    parquet_files,
    group_col=None,
    max_rank=7,
    n_files=None
):
    """
    Run complete reallocation analysis.

    Parameters
    ----------
    parquet_files : list
        List of synthetic population files.

    group_col : str, default None
        Segmentation variable.

    max_rank : int, default 7
        Maximum rank to include.

    n_files : int or None
        Number of files to process.
        None = all files.

    Returns
    -------
    summary : dict
        Structure:

        summary[group][rank] = {
            'mean': Series,
            'std' : Series,
            'n'   : int
        }
    """

    if n_files is not None:
        files = parquet_files[:n_files]
    else:
        files = parquet_files

    all_results = defaultdict(
        lambda: defaultdict(list)
    )

    # ----------------------------------
    # Process simulations
    # ----------------------------------

    for file in files:

        res = process_simulation(
            file,
            group_col=group_col,
            max_rank=max_rank
        )

        for group in res:

            for rank in res[group]:

                all_results[group][rank].append(
                    res[group][rank]
                )

    # ----------------------------------
    # Aggregate simulations
    # ----------------------------------

    summary = {}

    for group, rank_dict in all_results.items():

        summary[group] = {}

        for rank, simulations in rank_dict.items():

            df_rank = pd.concat(
                simulations,
                axis=1
            ).fillna(0)

            summary[group][rank] = {
                "mean": df_rank.mean(axis=1),
                "std": df_rank.std(axis=1),
                "n": len(simulations)
            }

    return summary

In [ ]:
def print_reallocation_summary(
    summary,
    include_ebike=True,
    show_total=True,
    title="GROUP"
):
    """
    Print reallocation tables for all groups.

    Parameters
    ----------
    summary : dict
        Output from run_reallocation_analysis()

    include_ebike : bool, default True
        Include E-bike stayers.

    show_total : bool, default True
        Print total contribution.

    title : str, default "GROUP"
        Header label (e.g. AGE GROUP, GENDER, INCOME).
    """

    for group in summary.keys():

        print("\n" + "=" * 60)
        print(f"{title}: {group}")
        print("=" * 60)

        table = summarize_reallocation(
            summary[group],
            include_ebike=include_ebike
        )

        print(table)

        if show_total:
            print(
                "\nTotal:",
                round(
                    table["Contribution (%)"].sum(),
                    2
                )
            )

### Age

In [ ]:
summary = run_reallocation_analysis(
    parquet_files,
    group_col="RespAgeCorrect",
    max_rank=7
)

print_reallocation_summary(
    summary,
    title="AGE GROUP",
    include_ebike=True
)

### Gender

In [ ]:
summary = run_reallocation_analysis(
    parquet_files,
    group_col="RespSex",
    max_rank=7
)

print_reallocation_summary(
    summary,
    title="RespSex",
    include_ebike=True
)

### Income

In [ ]:
summary = run_reallocation_analysis(
    parquet_files,
    group_col="IncFamily2023",
    max_rank=7
)

print_reallocation_summary(
    summary,
    title="IncFamily2023 GROUP",
    include_ebike=True
)

### Acces pop car

In [ ]:
summary = run_reallocation_analysis(
    parquet_files,
    group_col="access_pop_exp_car",
    max_rank=7
)

print_reallocation_summary(
    summary,
    title="Acce GROUP",
    include_ebike=True
)

### Car ownership

In [ ]:
summary = run_reallocation_analysis(
    parquet_files,
    group_col="HousehNumcars",
    max_rank=7
)

print_reallocation_summary(
    summary,
    title="HousehNumcars GROUP",
    include_ebike=True
)

### Purpose

In [ ]:
summary = run_reallocation_analysis(
    parquet_files,
    group_col="purposes",
    max_rank=7
)

print_reallocation_summary(
    summary,
    title="purposes GROUP",
    include_ebike=True
)